In [1]:
import os
import numpy as np
import cv2
import albumentations as albu
from glob import glob
from PIL import Image

def get_augmentation():
    transforms = [
        albu.HorizontalFlip(p=0.5),
        albu.VerticalFlip(p=0.5),
        albu.ShiftScaleRotate(
            shift_limit=0.1,
            scale_limit=0.1,
            rotate_limit=30,
            border_mode=cv2.BORDER_CONSTANT,
            p=0.5
        ),
        albu.RandomBrightnessContrast(contrast_limit=0.3,
                                      brightness_limit=0.3, p=0.3),
        albu.OneOf([
            albu.GaussNoise(p=1),
            albu.GaussianBlur(blur_limit=(3, 5), p=1),
            albu.ImageCompression(p=1),
        ], p=0.3),
    ]
    return albu.Compose(transforms, additional_targets={'mask': 'mask'})

data_dir = 'dataset_cellpose/train'               
aug_dir  = 'dataset_cellpose/train_aug'           

os.makedirs(aug_dir, exist_ok=True)

augment = get_augmentation()
n_augs_per_image = 4

image_paths = sorted(
    p for p in glob(os.path.join(data_dir, '*.png'))
    if '_mask' not in os.path.basename(p)
)

for img_path in image_paths:
    fname = os.path.basename(img_path)          
    base  = os.path.splitext(fname)[0]
    mask_name = f'{base}_mask.png'             
    mask_path = os.path.join(data_dir, mask_name)

    if not os.path.exists(mask_path):
        continue

    img  = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (512, 512), interpolation=cv2.INTER_LINEAR)
    mask = Image.open(mask_path)
    mask = np.array(mask).astype('uint8')
    mask = cv2.resize(mask, (512, 512), interpolation=cv2.INTER_NEAREST)

    cv2.imwrite(os.path.join(aug_dir, fname),
                cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
    cv2.imwrite(os.path.join(aug_dir, mask_name), mask)

    for i in range(n_augs_per_image):
        out = augment(image=img, mask=mask)
        img_a, mask_a = out['image'], out['mask']

        img_name_aug  = f'{base}_aug{i}.png'
        mask_name_aug = f'{base}_aug{i}_mask.png'

        cv2.imwrite(os.path.join(aug_dir, img_name_aug),
                    cv2.cvtColor(img_a, cv2.COLOR_RGB2BGR))
        cv2.imwrite(os.path.join(aug_dir, mask_name_aug), mask_a)

print("Done!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!")

/mnt/tank/scratch/plutskyi/anaconda3/envs/cellpose/lib/python3.11/site-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
